In [2]:
from datasets import load_dataset

# Wiki Mini – Wikipedia articles for text preprocessing and tokenization.
ds = load_dataset("gfhayworth/wiki_mini", split="train")
ds

Repo card metadata block was not found. Setting CardData to empty.


Dataset({
    features: ['psg'],
    num_rows: 169597
})

In [3]:
print("Number of articles:", len(ds))
print("Columns:", ds.column_names) # features of hugging face dataset 
print("First article:", ds[0])

Number of articles: 169597
Columns: ['psg']
First article: {'psg': 'Ted Cassidy (July 31, 1932 - January 16, 1979) was an American actor. He was best known for his roles as Lurch and Thing on "The Addams Family".'}


In [4]:
split = ds.train_test_split(test_size=0.1, seed=123)
train_ds, val_ds = split["train"], split["test"]

print("length of train", len(train_ds))
print("length of train", len(val_ds))

length of train 152637
length of train 16960


## Tokenize each passage and mark where it ends

In [5]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
EOT = tokenizer.eot_token

def tokenize(batch):
    ids = tokenizer.encode_ordinary_batch(batch["psg"])
    return {"ids" : [x + [EOT] for x in ids]}

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["psg"])
val_tok   = val_ds.map(tokenize, batched=True, remove_columns=["psg"])

In [6]:
# train_tok and val_tok are both Hugging Face Dataset objects, the same type as ds. The only difference is the column: psg (text) has been swapped for ids (token lists).
train_tok, val_tok

(Dataset({
     features: ['ids'],
     num_rows: 152637
 }),
 Dataset({
     features: ['ids'],
     num_rows: 16960
 }))

In [7]:
import numpy as np, torch

def flatten(tok_ds):
    return torch.from_numpy(np.concatenate(tok_ds["ids"]).astype(np.int64))

train_tokens = flatten(train_tok)
val_tokens   = flatten(val_tok)
train_tokens.shape, val_tokens.shape

(torch.Size([8639915]), torch.Size([960122]))

In [8]:
tokenizer.decode(train_tokens[:200].tolist())

'Aircraft hijacking (also known as skyjacking and aircraft piracy) is the takeover of an aircraft, by a person or group, usually armed. In most cases the pilot is forced to fly according to the orders of the hijackers. Sometimes, the hijackers fly the planes themselves, as believed to have occurred in the September 11, 2001 attacks. In one case the official pilot hijacked the plane, when he diverted his internal Air China flight to Taiwan.<|endoftext|>Jared Padalecki (born July 19, 1982) is an American actor. He is best known for his role on the television series "Gilmore Girls". Padalecki is also known for playing Sam Winchester on the CW television series "Supernatural".<|endoftext|>Viriato Clemente da Cruz, an Angolan poet and politician, was born in 1928 in Kikuvo (Porto Amboim), Angola and died in Beijing, China on 13 June 1973.<|endoftext|>Băileşti is a city'

Our GPTDatasetV1 builds every window up front and stores them in a list. With millions of tokens, that list gets slow to build and uses a lot of memory.

The better approach is to store only the flat tensor and cut each window when it's asked for.

In [9]:
from typing import Any

from torch.utils.data import Dataset, DataLoader

class TokenWindowDataset(Dataset):
    def __init__(self, tokens, max_length, stride) -> None:
        super().__init__()
        self.tokens = tokens
        self.max_length = max_length
        self.stride = stride

    def __len__(self):
        return (len(self.tokens) - self.max_length - 1) // self.stride + 1

    def __getitem__(self, index) -> Any:
        start = index * self.stride
        chunk = self.tokens[start : start + self.max_length + 1]
        return chunk[:-1], chunk[1:]

In [10]:
ctx = 256
train_loader = DataLoader(TokenWindowDataset(train_tokens, ctx, ctx),
                          batch_size=8, shuffle=True, drop_last=True)
val_loader = DataLoader(TokenWindowDataset(val_tokens, ctx, ctx),
                        batch_size=8, shuffle=False, drop_last=False)

In [11]:
x, y = next(iter(train_loader))
print(x.shape, y.shape)                  # torch.Size([8, 256]) twice
assert torch.equal(x[:, 1:], y[:, :-1])  # y really is x shifted by one
print(len(train_loader), len(val_loader))

torch.Size([8, 256]) torch.Size([8, 256])
4218 469


In [12]:
from helper_functions import choose_device
device = choose_device()

print(f"Using {device} device.")

Using mps device.


In [13]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}

## The learning-rate schedule

In [14]:
import math

def get_lr(step, max_lr, min_lr, warmup_steps, max_steps):
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    if step >= max_steps:
        return min_lr
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + 0.5 * (1 + math.cos(math.pi * progress)) * (max_lr - min_lr)


In [15]:
from helper_functions import calc_loss_batch, evaluate_model, generate_and_print_sample

def train_model(model, train_loader, val_loader, optimizer, device, tokenizer,
                max_steps, warmup_steps, max_lr, min_lr, eval_freq, eval_iter,
                start_context, ckpt_path = "../models/wiki_ckpt.pth", grad_clip = 1.0):
    
    train_losses, val_losses, track_tokens_seen, track_lrs = [], [], [], []
    tokens_seen, step = 0, 0
    best_val = float("inf")
    model.train()

    while step < max_steps:
        for input_batch, target_batch in train_loader:
            if step >= max_steps:
                break

            lr = get_lr(step, max_lr, min_lr, warmup_steps, max_steps)
            for group in optimizer.param_groups:
                group["lr"] = lr

            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            tokens_seen += input_batch.numel()
            track_lrs.append(lr)

            if step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"step {step:05d} | lr {lr:.2e} | train {train_loss:.3f} | val {val_loss:.3f}")

            step += 1


    generate_and_print_sample(model, tokenizer, device, start_context)
            
    return train_losses, val_losses, track_tokens_seen, track_lrs
            

In [16]:
from previous_chapters import GPTModel

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.1)

results = train_model(
    model, train_loader, val_loader, optimizer, device, tokenizer,
    max_steps=50, warmup_steps=10, max_lr=4e-4, min_lr=4e-5,
    eval_freq=10, eval_iter=5,
    start_context="The history of",
)

step 00000 | lr 4.00e-05 | train 10.778 | val 10.782
step 00010 | lr 4.00e-04 | train 8.168 | val 8.058
step 00020 | lr 3.47e-04 | train 7.378 | val 7.284
step 00030 | lr 2.20e-04 | train 7.190 | val 7.108
step 00040 | lr 9.27e-05 | train 7.150 | val 6.974
The history of the United.<|endoftext|>The.<|endoftext|>The the United, the United.<|endoftext|>The the United is a commun, the United of the United.<|endoftext|>The is a is a commun, and the United.<|endoftext|>The the United.<|endoftext|>The.<|endoftext|>The


In [21]:
from previous_chapters import generate
from helper_functions import text_to_token_ids, token_ids_to_text

model.eval()
token_ids = generate(
    model=model,
    idx=text_to_token_ids("The history of", tokenizer).to(device),
    max_new_tokens=50,
    context_size=GPT_CONFIG_124M["context_length"],
    temperature=0.8,
    top_k=40,
    eos_id=EOT,          # stop when the passage ends
)
print(token_ids_to_text(token_ids, tokenizer))

The history of the United. He was an American and the United of the first in the United.


In [22]:
torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    },
           "../models/wiki_mini_model.pth"
)